# SVM Loan Classification Project

## Import Libraries

In [ ]:
# ==========================================
# Import Required Libraries
# ==========================================

import warnings
warnings.filterwarnings("ignore")

# Data Handling
import numpy as np
import pandas as pd

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import ( StandardScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder)
# SVM
from sklearn.svm import SVC

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score)

# Save Model
import joblib


## Load Dataset

In [ ]:
df=pd.read_csv('loan_approved.csv')
df.head()

## EDA

In [ ]:
df.columns

In [ ]:
sns.countplot(x='Loan_Status (Approved)',data=df); plt.show()

In [ ]:
df.info()

In [ ]:
df.describe()


In [ ]:
df.describe(include=object)

In [ ]:
df.isnull().sum()

In [ ]:
missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": (df.isnull().sum()/len(df))*100
})
missing

In [ ]:
plt.figure(figsize=(10,5))
sns.heatmap(df.isnull(), cbar=False)
plt.title("Missing Values")
plt.show()

In [ ]:
print("Duplicate Rows :", df.duplicated().sum())

In [ ]:
df = df.drop_duplicates()

In [ ]:
# ==========================================
# Numerical and Categorical Columns
# ==========================================
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print("Numerical Columns")
print(numerical_cols)
print("\nCategorical Columns")
print(categorical_cols)

In [ ]:
# ==========================================
# Target Variable Distribution
# ==========================================

plt.figure(figsize=(6,4))
sns.countplot(
    data=df,
    x='Loan_Status (Approved)',
    palette='Set2'
)
plt.title("Loan Approval Distribution")
plt.xlabel("Loan Status")
plt.ylabel("Count")
plt.show()

In [ ]:
df["Loan_Status (Approved)"].value_counts(normalize=True)*100

In [ ]:
# ==========================================
# Histograms
# ==========================================
df[numerical_cols].hist(
    figsize=(14,8),
    bins=20,
    edgecolor="black"
)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# Boxplots
# ==========================================

plt.figure(figsize=(15,10))
for i, column in enumerate(numerical_cols):
    plt.subplot(2,3,i+1)
    sns.boxplot(y=df[column])
    plt.title(column)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# Countplots
# ==========================================
categorical_features = [
    "Gender",
    "Married",
    "Dependents",
    "Education",
    "Self_Employed",
    "Property_Area"
]
plt.figure(figsize=(15,12))
for i, column in enumerate(categorical_features):
    plt.subplot(3,2,i+1)
    sns.countplot(
        data=df,
        x=column,
        palette="Set3"
    )
    plt.xticks(rotation=20)

plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(
    data=df,
    x="Gender",
    hue="Loan_Status (Approved)"
)
plt.title("Loan Status by Gender")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(
    data=df,
    x="Education",
    hue="Loan_Status (Approved)"
)
plt.title("Loan Status by Education")
plt.show()

In [ ]:
plt.figure(figsize=(7,4))

sns.countplot(
    data=df,
    x="Property_Area",
    hue="Loan_Status (Approved)"
)

plt.title("Loan Status by Property Area")

plt.show()

In [ ]:
# ==========================================
# Correlation Heatmap
# ==========================================

plt.figure(figsize=(8,6))
sns.heatmap(
    df[numerical_cols].corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# ==========================================
# Pairplot
# ==========================================
selected_features = [
    "ApplicantIncome",
    "LoanAmount",
    "Credit_History",
    "Loan_Status (Approved)"
]
sns.pairplot(
    df[selected_features],
    hue="Loan_Status (Approved)"
)
plt.show()

In [ ]:
# ==========================================
# Features and Target
# ==========================================

X = df.drop("Loan_Status (Approved)", axis=1)

y = df["Loan_Status (Approved)"]

print("Features Shape :", X.shape)
print("Target Shape :", y.shape)

In [ ]:
y

In [ ]:
# ==========================================
# Encode Target Variable
# ==========================================
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)
print(label_encoder.classes_)

In [ ]:
y

In [ ]:
# ==========================================
# Feature Types
# ==========================================

ordinal_features = [
    "Dependents"
]
nominal_features = [
    "Gender",
    "Married",
    "Education",
    "Self_Employed",
    "Property_Area"
]
numerical_features = [
    "ApplicantIncome",
    "CoapplicantIncome",
    "LoanAmount",
    "Loan_Amount_Term",
    "Credit_History"
]
print("Ordinal Features")
print(ordinal_features)

print("\nNominal Features")
print(nominal_features)

print("\nNumerical Features")
print(numerical_features)

In [ ]:
categorical_cols = X.select_dtypes(include="object").columns

numerical_cols = X.select_dtypes(exclude="object").columns

In [ ]:
# ==========================================
# Numerical Pipeline
# ==========================================

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),   
    ("scaler", StandardScaler())
])

In [ ]:
# ==========================================
# Ordinal Pipeline
# ==========================================

ordinal_pipeline = Pipeline([   
    ("imputer", SimpleImputer(strategy="most_frequent")),   
    ("encoder",
     OrdinalEncoder(
         categories=[
             ["0","1","2","3+"]
         ]
     ))
])

In [ ]:
# ==========================================
# Nominal Pipeline
# ==========================================

nominal_pipeline = Pipeline([  
    ("imputer",
     SimpleImputer(strategy="most_frequent")),  
    ("encoder",
     OneHotEncoder(handle_unknown="ignore"))
])

## Preprocessing

In [ ]:
# ==========================================
# Column Transformer
# ==========================================

preprocessor = ColumnTransformer(   
    transformers=[       
        ("num",
         numerical_pipeline,
         numerical_features),     
        ("ord",
         ordinal_pipeline,
         ordinal_features),     
        ("nom",
         nominal_pipeline,
         nominal_features)  ])

In [ ]:
preprocessor

## Train/Test

In [ ]:
# ==========================================
# Train Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.30, random_state=42,stratify=y)
print("Training Shape :", X_train.shape)
print("Testing Shape :", X_test.shape)

In [ ]:
print(len(y_train[ y_train > 0]))
print(len(y_train[ y_train == 0]))

In [ ]:
print(len(y_test[ y_test > 0]))
print(len(y_test[ y_test == 0]))

In [ ]:
# ==========================================
# Fit Preprocessor
# ==========================================
preprocessor.fit(X_train)

In [ ]:
# ==========================================
# Save Transformer
# ==========================================

joblib.dump(
    preprocessor,
    "loan_preprocessor.pkl"
)

## Save Transformer

In [ ]:
# ==========================================
# Transform Data
# ==========================================
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [ ]:
print("Original Training Shape")
print(X_train.shape)
print()
print("Processed Training Shape")
print(X_train_processed.shape)

## Model

In [ ]:
# ==========================================
# Build SVM Classifier
# ==========================================

svm_model = SVC( kernel="rbf",  C=1, gamma="scale")

svm_model.fit(X_train_processed, y_train
)
print("Model Training Completed Successfully!")

In [ ]:
# ==========================================
# Predict Test Data
# ==========================================
y_pred = svm_model.predict(X_test_processed)
print(y_pred[:20])

In [ ]:
predictions = label_encoder.inverse_transform(y_pred)
predictions[:20]

In [ ]:
y_test.shape

In [ ]:
y_pred.shape

In [ ]:
accuracy = accuracy_score(  
    y_test,  
    y_pred)
print(f"Accuracy : {accuracy:.4f}")

print(f"Accuracy : {accuracy*100:.2f}%")

In [ ]:
cm = confusion_matrix(  
    y_test,
    y_pred
)
cm

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(  
    cm,  
    annot=True,
    
    cmap="Blues",
    
    fmt="d"
)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.title("Confusion Matrix")

plt.show()

In [ ]:
print(

classification_report(
    y_test,
    y_pred
)
)

In [ ]:
y_prob = svm_model.decision_function(  
    X_test_processed)

In [ ]:
fpr, tpr, thresholds = roc_curve(   
    y_test,   
    y_prob
)

auc_score = roc_auc_score(
    
    y_test,
    
    y_prob
)

In [ ]:
auc_score

In [ ]:
plt.figure(figsize=(6,5))

plt.plot( fpr,   tpr,  label=f"AUC = {auc_score:.3f}")
plt.plot(   [0,1], [0,1], 
    linestyle="--")

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()

plt.show()

In [ ]:
joblib.dump(svm_model, "loan_svm.pkl")

In [ ]:
# Grid Search CV 

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score

In [ ]:
params={'C':[0.1,1,10,15,35],'gamma':['scale',0.1,0.01,0.0005],'kernel':['linear','rbf','poly']}
grid=GridSearchCV(SVC(),params,cv=5)
grid.fit(X_train_processed,y_train)
best=grid.best_estimator_

In [ ]:
best

## Evaluation

In [ ]:
pred=best.predict(X_test_processed)
print(accuracy_score(y_test,pred))
print(confusion_matrix(y_test,pred))
print(classification_report(y_test,pred))

## Save Model

In [ ]:
joblib.dump(best,'svm_model.pkl')

## Predict New

In [ ]:
prep=joblib.load('preprocessor.pkl')
model=joblib.load('svm_model.pkl')
# new=pd.DataFrame([...])
# model.predict(prep.transform(new))